# IQM Pulse Tutorial

Pulse-level quantum programming with IQM Pulse.

**Documentation:** https://docs.meetiqm.com/iqm-pulse/

In [ ]:
from __future__ import annotations
import numpy as np
import matplotlib.pyplot as plt
from iqm.pulse.playlist.schedule import Schedule
from iqm.pulse.timebox import TimeBox, SchedulingAlgorithm

## Setup Lexis Session

In [ ]:
from py4lexis.session import LexisSession

lexis_session = LexisSession()
print("Lexis session initialized")

In [ ]:
from qaas import QProvider, QBackend

token = lexis_session.get_access_token()
lexis_project = "vlq_demo_project"
resource_name = "VLQ-CZ"

provider = QProvider(token, lexis_project, resource_name)

## Create ScheduleBuilder

ScheduleBuilder is the main interface for creating pulse schedules.

In [ ]:
# from iqm.pulse.builder import ScheduleBuilder

p = provider.get_pulla(resource_name)


In [ ]:
# Initialize schedule builder with quantum architecture
# Typically loaded from backend configuration
builder = p.get_schedule_builder()

print(f"ScheduleBuilder initialized")
print(f"Available qubits: {builder.chip_topology.qubits}")

## Build Simple Pulse Sequence

In [ ]:
# Create single-qubit gate operations
# PRX gate (parameterized X rotation)
rx_qb1 = builder.prx(["QB1"]).rx(np.pi/2)  # π/2 rotation around X

# SX gate (sqrt(X) gate)
sx_qb1 = builder.sx(["QB1"])

# RZ gate (virtual Z rotation)
rz_qb1 = builder.rz(["QB1"]).rz(np.pi/4)

print("Single-qubit gates created")
print(f"RX gate: {rx_qb1}")
print(f"SX gate: {sx_qb1}")
print(f"RZ gate: {rz_qb1}")

## Two-Qubit Gates

In [ ]:
# CZ gate (controlled-Z)
cz_gate = builder.cz(["QB1", "QB2"])

print(f"CZ gate: {cz_gate}")
print(f"Qubits involved: QB1, QB2")

## Measurement Operations

In [ ]:
# Measure operation
measure_op = builder.measure(["QB1", "QB2"])

print(f"Measure operation: {measure_op}")

## Build Complete Circuit

Combine operations to create a Bell state circuit.

In [ ]:
# Bell state: H(QB1) -> CZ(QB1, QB2) -> Measure

# Hadamard = RZ(π/2) + SX + RZ(π/2)
h_qb1 = (
    builder.rz(["QB1"]).rz(np.pi/2) +
    builder.sx(["QB1"]) +
    builder.rz(["QB1"]).rz(np.pi/2)
)

# CZ gate
cz_12 = builder.cz(["QB1", "QB2"])

# Measurement
meas = builder.measure(["QB1", "QB2"])

# Combine into circuit
bell_circuit = h_qb1 + cz_12 + meas

print("Bell state circuit created")
print(f"Number of operations: {len(bell_circuit)}")

## TimeBox Scheduling

TimeBoxes represent operations with timing constraints.

In [ ]:
# Create TimeBoxes for scheduling
timebox1 = TimeBox(
    operations=[rx_qb1],
    scheduling_algorithm=SchedulingAlgorithm.ASAP  # As Soon As Possible
)

timebox2 = TimeBox(
    operations=[sx_qb1],
    scheduling_algorithm=SchedulingAlgorithm.ALAP  # As Late As Possible
)

print(f"TimeBox 1 (ASAP): {timebox1}")
print(f"TimeBox 2 (ALAP): {timebox2}")

## Pulse Timing and Alignment

Control when operations execute relative to each other.

In [ ]:
# Parallel operations on different qubits
parallel_ops = (
    builder.prx(["QB1"]).rx(np.pi/2) +
    builder.prx(["QB2"]).rx(np.pi/2)
)

# Sequential operations on same qubit
sequential_ops = (
    builder.sx(["QB1"]) +
    builder.rz(["QB1"]).rz(np.pi/4) +
    builder.sx(["QB1"])
)

print("Parallel operations (QB1 and QB2 simultaneous)")
print("Sequential operations (QB1 gates in sequence)")

## Delay Operations

In [ ]:
# Add explicit delay
delay_op = builder.delay(["QB1"], duration=100e-9)  # 100 ns delay

# Circuit with delay
circuit_with_delay = (
    builder.sx(["QB1"]) +
    delay_op +
    builder.sx(["QB1"])
)

print(f"Delay operation: {delay_op}")
print(f"Duration: 100 ns")

## Conditional Operations (Fast Feedback)

In [ ]:
# Measure with feedback key
measure_feedback = builder.measure(["QB1"])(feedback_key="A")

# Conditional PRX based on measurement result
cc_prx = builder.cc_prx(["QB1"])(
    feedback_qubit="QB1",
    feedback_key="A"
)

# Reset operation (measure + conditional flip)
reset_op = builder.reset(["QB1"])

print("Conditional operations created")
print(f"Measure with feedback: {measure_feedback}")
print(f"Conditional PRX: {cc_prx}")

## GHZ State Circuit

In [ ]:
# 3-qubit GHZ state
# H(QB1) -> CZ(QB1,QB2) -> CZ(QB2,QB3)

h_qb1 = (
    builder.rz(["QB1"]).rz(np.pi/2) +
    builder.sx(["QB1"]) +
    builder.rz(["QB1"]).rz(np.pi/2)
)

cz_12 = builder.cz(["QB1", "QB2"])
cz_23 = builder.cz(["QB2", "QB3"])

meas_all = builder.measure(["QB1", "QB2", "QB3"])

ghz_circuit = h_qb1 + cz_12 + cz_23 + meas_all

print("GHZ state circuit created")
print(f"Operations: {len(ghz_circuit)}")

## Barrier Operation

In [ ]:
# Barrier prevents scheduling optimization across it
barrier = builder.barrier(["QB1", "QB2", "QB3"])

circuit_with_barrier = (
    builder.sx(["QB1"]) +
    builder.sx(["QB2"]) +
    barrier +
    builder.cz(["QB1", "QB2"])
)

print(f"Barrier operation: {barrier}")

## Visualize Circuit Timing

In [ ]:
def visualize_circuit_timing(circuit, title: str):
    """Simple visualization of circuit operations."""
    print(f"\n{title}")
    print("=" * 50)
    
    for idx, op in enumerate(circuit):
        print(f"{idx}: {op}")
    
    print(f"\nTotal operations: {len(circuit)}")

visualize_circuit_timing(bell_circuit, "Bell State Circuit")
visualize_circuit_timing(ghz_circuit, "GHZ State Circuit")

## Custom Gate Implementation Example

In [ ]:
# Custom rotation sequence
def custom_rotation(builder, qubit: str, angle: float):
    """Custom rotation combining RZ and SX gates."""
    return (
        builder.rz([qubit]).rz(angle) +
        builder.sx([qubit]) +
        builder.rz([qubit]).rz(-angle)
    )

# Use custom rotation
custom_gate = custom_rotation(builder, "QB1", np.pi/3)

print("Custom rotation gate created")
print(f"Operations: {len(custom_gate)}")